In [18]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,f1_score,auc,roc_auc_score,roc_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.multioutput import MultiOutputClassifier

In [19]:
train_info = pd.read_csv("train_info.csv")
test_info = pd.read_csv("test_info.csv")
test_answer = pd.read_csv("test_answer.csv")

In [20]:
def chenge(x):
    return [int(i) for i in str(x).replace('[','').replace(']','').split() if i.strip() != '']

In [21]:
train_info['cut_point'] = train_info['cut_point'].apply(chenge)
test_info['cut_point'] = test_info['cut_point'].apply(chenge)

In [22]:
data_1 = []
data_2 = []

In [23]:
for idx, row in train_info.iterrows():
    unique_id = row['unique_id']
    cut_points = row['cut_point']
    mode = row['mode']
    file_path = f"train_data/{unique_id}.txt"

    data_text = pd.read_csv(file_path, header=None, sep=r'\s+', names=['Ax','Ay','Az','Gx','Gy','Gz'])
    data_text = data_text.apply(pd.to_numeric, errors='coerce').dropna()

    p = 0
    for cp in cut_points:
        segment = data_text.iloc[p:cp]
        p = cp
        if len(segment) == 0:
            continue

        segment_mean = segment.mean()
        segment_std = segment.std()
        features = pd.concat([segment_mean, segment_std]).tolist()

        data_1.append({
            'unique_id': unique_id,
            'mode': mode,
            'play years': row['play years'],
            'hold racket handed': row['hold racket handed'],
            'level': row['level'],
            'gender': row['gender'],
            **{f'feat{i}': features[i] for i in range(len(features))}
        })


In [24]:
for idx, row in test_info.iterrows():
    unique_id2 = row['unique_id']
    cut_points2 = row['cut_point']
    mode2 = row['mode']
    file_path2 = f"test_data/{unique_id2}.txt"

    data_text2 = pd.read_csv(file_path2, header=None, sep=r'\s+', names=['Ax','Ay','Az','Gx','Gy','Gz'])
    data_text2 = data_text2.apply(pd.to_numeric, errors='coerce').dropna()

    p2 = 0
    for cp2 in cut_points2:
        segment2 = data_text2.iloc[p2:cp2]
        p2 = cp2
        if len(segment2) == 0:
            continue

        segment_mean2 = segment2.mean()
        segment_std2 = segment2.std()
        features2 = pd.concat([segment_mean2, segment_std2]).tolist()

        data_2.append({
            'unique_id': unique_id2,
            'mode': mode2,
            **{f'feat{i}': features2[i] for i in range(len(features2))}
        })


In [25]:
final_df = pd.DataFrame(data_1)
final_df2 = pd.DataFrame(data_2)

In [26]:
X = final_df[[col for col in final_df.columns if col.startswith('feat')] + ['mode']].copy()
if X['mode'].dtype == 'object':
    X['mode'] = X['mode'].astype('category').cat.codes

In [27]:
y = final_df[['play years','hold racket handed','level','gender']].copy()
label_encoders = {}

In [28]:
for col in y.columns:
    if y[col].dtype == 'object':
        le = LabelEncoder()
        y[col] = le.fit_transform(y[col])
        label_encoders[col] = le

In [29]:
X2 = final_df2[[col for col in final_df2.columns if col.startswith('feat')] + ['mode']].copy()
if X2['mode'].dtype == 'object':
    X2['mode'] = X2['mode'].astype('category').cat.codes

In [30]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X2_scaled = scaler.transform(X2)

In [31]:
knn = KNeighborsClassifier(n_neighbors=5)
clf = MultiOutputClassifier(knn)
clf.fit(X_scaled, y)
y_pred3 = clf.predict(X2_scaled)

In [32]:
pred_df = final_df2[['unique_id']].copy()
for i, col in enumerate(y.columns):
    pred_df[col] = y_pred3[:, i]

In [33]:
agg_pred = pred_df.groupby('unique_id').agg(lambda x: x.mode()[0]).reset_index()

merged = pd.merge(test_answer, agg_pred, on='unique_id', suffixes=('_true', '_pred'))

for col in y.columns:
    print(f"{col} Accuracy:", accuracy_score(merged[f"{col}_true"], merged[f"{col}_pred"]))
    print(f"{col} f1:", f1_score(merged[f"{col}_true"], merged[f"{col}_pred"],average =  'weighted',labels = train_info[col].unique()))

play years Accuracy: 0.3958041958041958
play years f1: 0.3020476071519965
hold racket handed Accuracy: 0.9965034965034965
hold racket handed f1: 0.9964866179618256
level Accuracy: 0.4489510489510489
level f1: 0.36657310841565227
gender Accuracy: 0.7916083916083916
gender f1: 0.7606055140088054


In [34]:
fpr, tpr, _ = roc_curve(merged["hold racket handed_true"].values - 1,merged["hold racket handed_pred"].values - 1)
roc_auc = auc(fpr, tpr)
print(roc_auc)

0.9884259259259259


In [35]:
fpr, tpr, _ = roc_curve(merged["gender_true"].values - 1,merged["gender_pred"].values - 1)
roc_auc = auc(fpr, tpr)
print(roc_auc)

0.6077828054298643
